In [60]:
import pandas as pd
import os
import json
import numpy as np
from os.path import dirname

root_path = dirname(os.getcwd())

pd.set_option("display.max_columns", None)
data_dir = root_path + "/data/datasets/original/"
data_dir_processed = root_path + "/data/datasets/processed/"
data_dir_graphs = root_path + "/data/datasets/graphs_repair/"

print(root_path, data_dir, data_dir_processed, data_dir_graphs, sep="\n")

/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/original/
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/processed/
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/graphs_repair/


In [61]:
with open("dataset_features.json", 'r') as file:
    datasets_info = json.load(file)


In [62]:
list(datasets_info.keys())

['BPI12_DECLINED_COMPLETE',
 'sepsis_cases_1',
 'sepsis_cases_4',
 'BPIC15_common',
 'BPIC15_4_f2',
 'bpic2012_O_ACCEPTED-COMPLETE']

In [63]:
dataset = "BPI12_DECLINED_COMPLETE"

In [64]:
tab_all = pd.read_csv(f"datasets/processed/{dataset}_processed_all.csv")
tab_all.head()

,Resource,lifecycle:transition,timesincemidnight,timesincelastevent,timesincecasestart,event_nr,month,weekday,hour,open_cases,time:timestamp,Activity,case:AMOUNT_REQ,CaseID,case:label,remaining_time
0,0.0,SCHEDULE,941.0,6.113517,24061.486817,46.0,11.0,4.0,15.0,758.0,1.321631e+09,wwijzigencontractgegevensschedule,25000.0,181447,False,1238.557
1,0.0,SCHEDULE,942.0,0.741917,24062.228733,47.0,11.0,4.0,15.0,758.0,1.321631e+09,wwijzigencontractgegevensschedule,25000.0,181447,False,1194.042
2,0.0,START,899.0,11766.656217,12505.183217,13.0,11.0,3.0,14.0,737.0,1.321542e+09,wnabellenoffertesstart,5000.0,183277,False,1831084.301
3,0.0,COMPLETE,435.0,975.723167,13480.906383,14.0,11.0,4.0,7.0,788.0,1.321601e+09,wnabellenoffertescomplete,5000.0,183277,False,1772540.911
4,0.0,START,570.0,15.111433,61129.463333,29.0,12.0,2.0,9.0,646.0,1.324460e+09,wvaliderenaanvraagstart,12850.0,183280,False,518878.255


In [65]:
tab_train = pd.read_csv(f"datasets/processed/{dataset}_processed_train.csv")
tab_valid = pd.read_csv(f"datasets/processed/{dataset}_processed_valid.csv")
tab_test = pd.read_csv(f"datasets/processed/{dataset}_processed_test.csv")

In [66]:
if dataset == "BPIC15_4_f2":
    with open("dataset_features.json", 'r') as file:
        dataset_info = json.load(file)[dataset]
elif dataset.startswith("BPIC15"):
    with open("dataset_features.json", 'r') as file:
        dataset_info = json.load(file)["BPIC15_common"]
else:
    with open("dataset_features.json", 'r') as file:
        dataset_info = json.load(file)[dataset]

In [67]:
dataset_info

{'categorical': ['Resource', 'lifecycle:transition', 'Activity'],
 'numerical': ['timesincemidnight',
  'timesincelastevent',
  'timesincecasestart',
  'event_nr',
  'month',
  'weekday',
  'hour',
  'open_cases',
  'time:timestamp',
  'case:AMOUNT_REQ']}

In [68]:
categorical_columns = dataset_info["categorical"]
real_value_columns = dataset_info["numerical"]

In [69]:
'''
for k in categorical_columns:
    tab_all[k] = tab_all[k].astype("object")
    tab_train[k] = tab_train[k].astype("object")
    tab_valid[k] = tab_valid[k].astype("object")
    tab_test[k] = tab_test[k].astype("object")
''' 

for k in categorical_columns:
    tab_all[k] = tab_all[k].astype(str)
    tab_train[k] = tab_train[k].astype(str)
    tab_valid[k] = tab_valid[k].astype(str)
    tab_test[k] = tab_test[k].astype(str)

### Prepare the graphs

In [70]:
import sklearn.preprocessing

from typing import List

In [71]:
def get_case_ids(tab):
    return list(tab["CaseID"].unique())

In [72]:
from torch import tensor, max, int64, float32
from torch_geometric.data import HeteroData

In [73]:
def get_one_hot_encoder(dataset: pd.DataFrame, key: str):
    datas = dataset[key].unique()
    datas = datas.reshape([len(datas), 1])
    onehot = sklearn.preprocessing.OneHotEncoder()
    onehot.fit(datas)
    return onehot

In [74]:
def get_one_hot_encodings(
    onehot, datas: pd.Series
):
    return onehot.transform(datas.reshape(-1, 1)).toarray()

In [75]:
def get_node_features(dataset: pd.DataFrame, trace: pd.DataFrame, cat_features, real_features) -> dict:
 

    res = {}

    for key in trace:
        values = trace[key].values
        if key in cat_features:
            onehot_encoder = get_one_hot_encoder(dataset, key)
            try:
                res[key] = tensor(
                    get_one_hot_encodings(onehot_encoder, values),
                    dtype=float32,
                    requires_grad=True
                )
            except ValueError:
                print(key)
                print(values)
        if key in real_features:
            res[key] = tensor(values,  dtype=float32,requires_grad=True)
            res[key] = res[key].reshape(res[key].shape[0], 1)
        
    

    return res


In [76]:


def compute_edges_indexs(node_features: dict, prefix_len):
    res = {}
    keys = node_features.keys()
    
    indexes = [[i, i + 1] for i in range(prefix_len-1)]
   
    for k in keys:
        if len(node_features[k]) != 1:
            if k == "Activity":
                res[(k, "followed_by", k)] = indexes
                for k2 in keys:
                    if k2 != k:
                        if len(node_features[k2]) == 1:
                            res[(k, "related_to", k2)] = [
                                [i, 0] for i in range(prefix_len)
                            ]
                        else:
                            res[(k, "related_to", k2)] = [
                                [i, i] for i in range(prefix_len)
                            ]
            else:
                res[(k, "related_to", k)] = indexes

    return res

In [77]:



def build_prefixes_graph_from_trace(dataset, trace, cat_features, real_features, prefix_length):
    X = []  # graphs
   
    
    
    node_features = get_node_features(dataset, trace, cat_features, real_features)
    
    
    
    
    G = HeteroData()
        
        
        
    for k in node_features:
        if k != "case:label":
            G[k].x = node_features[k][:prefix_length]


    edges_indexes = compute_edges_indexs(node_features, prefix_length)

    


    for k in edges_indexes:
        ce = [[], []]
        for i in range(len(edges_indexes[k])):
            ce[0].append(edges_indexes[k][i][0])
            ce[1].append(edges_indexes[k][i][1])
        edges_indexes[k] = ce

    for k in edges_indexes:
        G[k].edge_index = tensor(edges_indexes[k], dtype=int64)


    ## Get the label of the trace
    label_value = trace["remaining_time"].values[prefix_length -1]
    G.y = tensor([label_value], dtype=float32)
    
        
    X.append(G)
    
    return X

## Create the datasets

In [78]:
case_train_ids = get_case_ids(tab_train)
case_valid_ids = get_case_ids(tab_valid)
case_test_ids = get_case_ids(tab_test)

In [79]:
print(len(case_train_ids))
print(len(case_valid_ids))
print(len(case_test_ids))

2998
750
937


In [80]:
tab_train["CaseID"] = tab_train["CaseID"].astype(np.str_)
tab_valid["CaseID"] = tab_valid["CaseID"].astype(np.str_)
tab_test["CaseID"] = tab_test["CaseID"].astype(np.str_)

In [81]:
trace = (
        tab_train.query(f"CaseID == '{case_train_ids[0]}'")
        .reset_index()
        .drop(columns="index")
        .drop(columns="CaseID")
    )
trace 

,Resource,lifecycle:transition,timesincemidnight,timesincelastevent,timesincecasestart,event_nr,month,weekday,hour,open_cases,time:timestamp,Activity,case:AMOUNT_REQ,case:label,remaining_time
0,0.0,SCHEDULE,941.0,6.113517,24061.486817,46.0,11.0,4.0,15.0,758.0,1.321631e+09,wwijzigencontractgegevensschedule,25000.0,False,1238.557
1,0.0,SCHEDULE,942.0,0.741917,24062.228733,47.0,11.0,4.0,15.0,758.0,1.321631e+09,wwijzigencontractgegevensschedule,25000.0,False,1194.042
2,11189.0,START,739.0,92.035750,819.520133,7.0,11.0,2.0,12.0,686.0,1.320236e+09,wcompleterenaanvraagstart,25000.0,False,1395756.558
3,11189.0,COMPLETE,752.0,12.639017,832.159150,8.0,11.0,2.0,12.0,688.0,1.320237e+09,aacceptedcomplete,25000.0,False,1394998.217
4,11189.0,COMPLETE,755.0,0.000000,834.909267,9.0,11.0,2.0,12.0,688.0,1.320237e+09,afinalizedcomplete,25000.0,False,1394833.210
5,11189.0,COMPLETE,755.0,2.750117,834.909267,10.0,11.0,2.0,12.0,688.0,1.320237e+09,oselectedcomplete,25000.0,False,1394833.210
6,11189.0,COMPLETE,755.0,0.023867,834.933133,11.0,11.0,2.0,12.0,688.0,1.320237e+09,ocreatedcomplete,25000.0,False,1394831.778
7,11189.0,COMPLETE,755.0,0.000617,834.933750,12.0,11.0,2.0,12.0,688.0,1.320237e+09,osentcomplete,25000.0,False,1394831.741
8,11189.0,SCHEDULE,755.0,0.002383,834.936133,13.0,11.0,2.0,12.0,688.0,1.320237e+09,wnabellenoffertesschedule,25000.0,False,1394831.598
9,11189.0,COMPLETE,755.0,0.029217,834.965350,14.0,11.0,2.0,12.0,688.0,1.320237e+09,wcompleterenaanvraagcomplete,25000.0,False,1394829.845


In [82]:
min_len = tab_all.groupby("CaseID").size().min()
max_len = tab_all.groupby("CaseID").size().max()
print("Minimum trace length:", min_len)
print("Maximum trace length:", max_len)

Minimum trace length: 15
Maximum trace length: 175


In [83]:
import pickle
from tqdm.notebook import tqdm

In [84]:
PREFIX_LENGTH = 4

In [85]:
print("Preparing training dataset...")

X_train = []


for i in tqdm(range(len(case_train_ids))):
    trace = (
        tab_train.query(f"CaseID == '{case_train_ids[i]}'")
        .reset_index(drop=True)
        .drop(columns="CaseID")
    )

    if len(trace) >= PREFIX_LENGTH:
        graphs = build_prefixes_graph_from_trace(
            dataset=tab_all,
            trace=trace,
            cat_features=categorical_columns,
            real_features=real_value_columns,
            prefix_length=PREFIX_LENGTH,
        )
        for j in range(len(graphs)):
            X_train.append(graphs[j])

Preparing training dataset...


  0%|          | 0/2998 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
with open(data_dir_graphs + dataset + "_TRAIN_repair.pkl", "wb") as f:
    pickle.dump(X_train, f)

In [ ]:
print("Preparing validation dataset...")

X_valid = []


for i in tqdm(range(len(case_valid_ids))):
    trace = (
        tab_valid.query(f"CaseID == '{case_valid_ids[i]}'")
        .reset_index(drop=True)
        .drop(columns="CaseID")
    )
    if len(trace) >= PREFIX_LENGTH:
        graphs = build_prefixes_graph_from_trace(
            dataset=tab_all,
            trace=trace,
            cat_features=categorical_columns,
            real_features=real_value_columns,
            prefix_length=PREFIX_LENGTH
        )
        for j in range(len(graphs)):
            X_valid.append(graphs[j])

Preparing validation dataset...


  0%|          | 0/750 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
with open(data_dir_graphs + dataset + "_VALID_repair.pkl", "wb") as f:
    pickle.dump(X_valid, f)

In [ ]:
X_tests={}
MaxPrefix = 20

for L in range(1,MaxPrefix+1):
    print(f"Preparing test dataset {L}...")
    X_test_L = []
    
    for i in tqdm(range(len(case_test_ids))):
        trace = (
            tab_test.query(f"CaseID == '{case_test_ids[i]}'")
            .reset_index()
            .drop(columns="index")
            .drop(columns="CaseID")
        )
        
        if len(trace) > L:  #generate_prefix_data in experiments/DatasetManager confirms >=
            graphs = build_prefixes_graph_from_trace(
                dataset=tab_all,
                trace=trace,
                cat_features=categorical_columns,
                real_features=real_value_columns,
                prefix_length=L
            )
            X_test_L.extend(graphs)
        
    X_tests[L] = X_test_L
        

Preparing test dataset 1...


  0%|          | 0/937 [00:00<?, ?it/s]

Preparing test dataset 2...


  0%|          | 0/937 [00:00<?, ?it/s]

Preparing test dataset 3...


...

  0%|          | 0/937 [00:00<?, ?it/s]

Preparing test dataset 19...


  0%|          | 0/937 [00:00<?, ?it/s]

Preparing test dataset 20...


  0%|          | 0/937 [00:00<?, ?it/s]

In [ ]:
#with open(data_dir_graphs + dataset + "_TEST_repair.pkl", "wb") as f:
#    pickle.dump(X_test, f)


for L, X_test_L in X_tests.items():
    fname   = f"{dataset}_TEST{L}_repair.pkl"
    outpath = os.path.join(data_dir_graphs, fname)
    with open(outpath, "wb") as f:
        pickle.dump(X_test_L, f)
    print(f"Saved {len(X_test_L)} graphs for prefix {L} to {outpath}")

Saved 937 graphs for prefix 1 to /home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/graphs_repair/bpic2012_O_ACCEPTED-COMPLETE_TEST1_repair.pkl
Saved 937 graphs for prefix 2 to /home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/graphs_repair/bpic2012_O_ACCEPTED-COMPLETE_TEST2_repair.pkl
Saved 937 graphs for prefix 3 to /home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/graphs_repair/bpic2012_O_ACCEPTED-COMPLETE_TEST3_repair.pkl
Saved 937 graphs for prefix 4 to /home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/graphs_repair/bpic2012_O_ACCEPTED-COMPLETE_TEST4_repair.pkl
Saved 937 graphs for prefix 5 to /home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/graphs_repair/bpic2012_O_ACCEPTED-COMPLETE_TEST5_repair.pkl
Saved 937 graphs for prefix 6 to /home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/graphs_repair/bpic2012_O_ACCEPTED-COMPLETE_TEST6_repair.pkl
Saved 937 graphs for prefix 